<a href="https://colab.research.google.com/github/EngrOtee/LLM-Based-Multilingual-Sentiment-Analysis-for-Tourism-with-Low-Resource-Languages/blob/main/LLM_DATASET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load Nepal
nepal = pd.read_csv('nepal_tourism_reviews.csv')

# Keep only rows where label is positive or negative — drop neutral
nepal_binary = nepal[nepal['label'].isin(['positive', 'negative'])].copy()

# Standardize into our unified schema
nepal_clean = pd.DataFrame({
    'text': nepal_binary['review'],
    'label': nepal_binary['label'],           # already 'positive'/'negative' strings
    'source_dataset': 'nepal',
    'language': 'english',
    'domain': nepal_binary['domain'].str.lower()  # 'attraction' or 'hotel'
})

print(f"Original Nepal rows: {len(nepal)}")
print(f"After dropping neutral: {len(nepal_clean)}")
print(nepal_clean['label'].value_counts())
print(nepal_clean.head())

Original Nepal rows: 10536
After dropping neutral: 7637
label
positive    6274
negative    1363
Name: count, dtype: int64
                                                text     label source_dataset  \
0  We visited Devi's Falls and honestly regretted...  negative          nepal   
1  VisitorsVisitors pay a small fee, then enter t...  negative          nepal   
3  overrated. nothing much about the devi's fall....  negative          nepal   
4  Nothing here to see that is memorable. This is...  negative          nepal   
6  This was a complete waste of time. Hardly any ...  negative          nepal   

  language      domain  
0  english  attraction  
1  english  attraction  
3  english  attraction  
4  english  attraction  
6  english  attraction  


In [ ]:
# Load all three KazSAnDRA splits
kaz_train = pd.read_csv('01_pc_train_ib_excel.csv')
kaz_valid = pd.read_csv('04_pc_valid_excel.csv')
kaz_test = pd.read_csv('05_pc_test_excel.csv')

# Combine them into one dataframe — we'll re-split train/valid/test ourselves later,
# after merging with the other datasets, so it's a consistent split across everything
kaz_all = pd.concat([kaz_train, kaz_valid, kaz_test], ignore_index=True)

# Map 0/1 to word labels, matching Nepal's scheme
label_map = {0: 'negative', 1: 'positive'}
kaz_all['label_mapped'] = kaz_all['label'].map(label_map)

# Standardize into our unified schema
kaz_clean = pd.DataFrame({
    'text': kaz_all['text_cleaned'],   # using the pre-cleaned text column
    'label': kaz_all['label_mapped'],
    'source_dataset': 'kazsandra',
    'language': 'kazakh',
    'domain': kaz_all['domain']        # appstore/market/mapping/bookstore
})

print(f"Total KazSAnDRA rows: {len(kaz_clean)}")
print(kaz_clean['label'].value_counts())
print(kaz_clean.head())

Total KazSAnDRA rows: 167961
label
positive    138022
negative     29939
Name: count, dtype: int64
                                     text     label source_dataset language  \
0                              өтте күшті  positive      kazsandra   kazakh   
1  мәбазар жок оте керемет тамаша керемет  positive      kazsandra   kazakh   
2                   кушти дал тура айтады  positive      kazsandra   kazakh   
3                             реклама коп  negative      kazsandra   kazakh   
4                          5 баға беремін  positive      kazsandra   kazakh   

     domain  
0  appstore  
1  appstore  
2  appstore  
3  appstore  
4  appstore  


In [ ]:
# Load all three NaijaSenti Pidgin splits
pcm_train = pd.read_csv('train_pcm.csv')
pcm_dev = pd.read_csv('dev_pcm.csv')
pcm_test = pd.read_csv('test_pcm.csv')

# Combine into one dataframe, same reasoning as KazSAnDRA — re-split later, consistently
pcm_all = pd.concat([pcm_train, pcm_dev, pcm_test], ignore_index=True)

# Drop neutral, keep only positive/negative
pcm_binary = pcm_all[pcm_all['label'].isin(['positive', 'negative'])].copy()

# Standardize into our unified schema
pcm_clean = pd.DataFrame({
    'text': pcm_binary['tweet'],
    'label': pcm_binary['label'],
    'source_dataset': 'naijasenti',
    'language': 'pidgin',
    'domain': 'twitter'
})

print(f"Original NaijaSenti (pcm) rows: {len(pcm_all)}")
print(f"After dropping neutral: {len(pcm_clean)}")
print(pcm_clean['label'].value_counts())
print(pcm_clean.head())

Original NaijaSenti (pcm) rows: 10556
After dropping neutral: 10032
label
negative    6380
positive    3652
Name: count, dtype: int64
                                                text     label source_dataset  \
0  yeah ‍️the guy wants to trend dat was why e jo...  negative     naijasenti   
1  this life is so funny sef you will work hard a...  negative     naijasenti   
2  dis is unfair of urcompany goingtowks nw ive b...  negative     naijasenti   
3  lil wayne don vex me im actually not excited a...  negative     naijasenti   
4  dis is unfair of ur company going to wks nw iv...  negative     naijasenti   

  language   domain  
0   pidgin  twitter  
1   pidgin  twitter  
2   pidgin  twitter  
3   pidgin  twitter  
4   pidgin  twitter  


In [ ]:
# Stack all three cleaned dataframes into one
all_data = pd.concat([nepal_clean, kaz_clean, pcm_clean], ignore_index=True)

print(f"Total combined rows: {len(all_data)}")
print()
print("By source_dataset:")
print(all_data['source_dataset'].value_counts())
print()
print("By label (overall):")
print(all_data['label'].value_counts())
print()
print("Label balance WITHIN each source (important — check this):")
print(pd.crosstab(all_data['source_dataset'], all_data['label']))
print()
print("Any missing/null text?", all_data['text'].isna().sum())
print("Any empty-string text?", (all_data['text'].str.strip() == '').sum())

Total combined rows: 185630

By source_dataset:
source_dataset
kazsandra     167961
naijasenti     10032
nepal           7637
Name: count, dtype: int64

By label (overall):
label
positive    147948
negative     37682
Name: count, dtype: int64

Label balance WITHIN each source (important — check this):
label           negative  positive
source_dataset                    
kazsandra          29939    138022
naijasenti          6380      3652
nepal               1363      6274

Any missing/null text? 0
Any empty-string text? 0


In [ ]:
from sklearn.model_selection import train_test_split

# ---- STAGE 1 DATA: language adaptation (Kazakh + Pidgin) ----
adaptation_data = pd.concat([kaz_clean, pcm_clean], ignore_index=True)

print("=== ADAPTATION DATA (Stage 1) ===")
print(f"Total rows: {len(adaptation_data)}")
print(adaptation_data['source_dataset'].value_counts())
print(adaptation_data['label'].value_counts())
print()

# ---- STAGE 2 DATA: tourism task (Nepal only) — split into train/valid/test ----
# First split: 70% train, 30% temp (which we'll split again into valid/test)
nepal_train, nepal_temp = train_test_split(
    nepal_clean,
    test_size=0.3,
    random_state=42,           # fixes the random shuffle so results are reproducible
    stratify=nepal_clean['label']  # keeps the 82/18 pos/neg ratio consistent across splits
)

# Second split: divide the 30% temp into 15% valid, 15% test
nepal_valid, nepal_test = train_test_split(
    nepal_temp,
    test_size=0.5,
    random_state=42,
    stratify=nepal_temp['label']
)

print("=== TOURISM DATA (Stage 2) — Nepal only ===")
print(f"Train: {len(nepal_train)} rows")
print(f"Valid: {len(nepal_valid)} rows")
print(f"Test:  {len(nepal_test)} rows")
print()
print("Checking label ratio held across splits (should all be ~82% positive):")
for name, df in [('train', nepal_train), ('valid', nepal_valid), ('test', nepal_test)]:
    pct_pos = (df['label'] == 'positive').mean() * 100
    print(f"  {name}: {pct_pos:.1f}% positive ({len(df)} rows)")


=== ADAPTATION DATA (Stage 1) ===
Total rows: 177993
source_dataset
kazsandra     167961
naijasenti     10032
Name: count, dtype: int64
label
positive    141674
negative     36319
Name: count, dtype: int64

=== TOURISM DATA (Stage 2) — Nepal only ===
Train: 5345 rows
Valid: 1146 rows
Test:  1146 rows

Checking label ratio held across splits (should all be ~82% positive):
  train: 82.2% positive (5345 rows)
  valid: 82.2% positive (1146 rows)
  test: 82.1% positive (1146 rows)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Convert text to TF-IDF features — this turns each review into a vector of word importance scores
vectorizer = TfidfVectorizer(
    max_features=5000,      # keep only the 5000 most informative words/phrases
    ngram_range=(1, 2),     # use both single words and two-word phrases (bigrams)
    stop_words='english'    # drop common words like "the", "and", "is"
)

X_train = vectorizer.fit_transform(nepal_train['text'])
X_test = vectorizer.transform(nepal_test['text'])

y_train = nepal_train['label']
y_test = nepal_test['label']

# Train Logistic Regression with class weighting to handle the 82/18 imbalance
baseline_model = LogisticRegression(
    class_weight='balanced',   # automatically up-weights the minority (negative) class
    max_iter=1000,
    random_state=42
)
baseline_model.fit(X_train, y_train)

# Evaluate
y_pred = baseline_model.predict(X_test)

print("=== BASELINE (TF-IDF + Logistic Regression) — Nepal test set ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred, average='weighted'):.3f}")
print(f"F1 (macro): {f1_score(y_test, y_pred, average='macro'):.3f}")
print()
print(classification_report(y_test, y_pred))

=== BASELINE (TF-IDF + Logistic Regression) — Nepal test set ===
Accuracy: 0.925
F1 (weighted): 0.926
F1 (macro): 0.875

              precision    recall  f1-score   support

    negative       0.77      0.82      0.80       205
    positive       0.96      0.95      0.95       941

    accuracy                           0.92      1146
   macro avg       0.87      0.88      0.88      1146
weighted avg       0.93      0.92      0.93      1146



In [ ]:
!pip install transformers datasets torch scikit-learn -q

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ---- 1. Prepare labels as integers (models need numbers, not words) ----
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

nepal_train_hf = nepal_train.copy()
nepal_valid_hf = nepal_valid.copy()
nepal_test_hf = nepal_test.copy()

for df in [nepal_train_hf, nepal_valid_hf, nepal_test_hf]:
    df['label_id'] = df['label'].map(label2id)

# ---- 2. Convert pandas dataframes to HuggingFace Dataset objects ----
train_ds = Dataset.from_pandas(nepal_train_hf[['text', 'label_id']])
valid_ds = Dataset.from_pandas(nepal_valid_hf[['text', 'label_id']])
test_ds = Dataset.from_pandas(nepal_test_hf[['text', 'label_id']])

# ---- 3. Load tokenizer and tokenize the text ----
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_ds = train_ds.map(tokenize_fn, batched=True)
valid_ds = valid_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

# Rename label_id -> labels (HuggingFace's Trainer expects this exact name)
train_ds = train_ds.rename_column('label_id', 'labels')
valid_ds = valid_ds.rename_column('label_id', 'labels')
test_ds = test_ds.rename_column('label_id', 'labels')

train_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
valid_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# ---- 4. Compute class weights (same reasoning as the baseline's class_weight='balanced') ----
class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=nepal_train_hf['label_id'].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"Class weights (negative, positive): {class_weights}")

# ---- 5. Load model ----
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

# ---- 6. Custom Trainer that applies class weighting in the loss ----
# (Standard Trainer doesn't support class_weight directly like sklearn does,
# so we override compute_loss to plug it in ourselves.)
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# ---- 7. Metrics function — same metrics as the baseline, for a fair comparison ----
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

# ---- 8. Training arguments ----
training_args = TrainingArguments(
    output_dir='./xlmr_nepal',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    compute_metrics=compute_metrics,
)

# ---- 9. Train ----
trainer.train()

# ---- 10. Evaluate on the held-out test set (same set the baseline used) ----
test_results = trainer.predict(test_ds)
preds = np.argmax(test_results.predictions, axis=-1)
labels = test_results.label_ids

print("\n=== XLM-R FINE-TUNED — Nepal test set ===")
print(f"Accuracy: {accuracy_score(labels, preds):.3f}")
print(f"F1 (weighted): {f1_score(labels, preds, average='weighted'):.3f}")
print(f"F1 (macro): {f1_score(labels, preds, average='macro'):.3f}")
print()
print(classification_report(labels, preds, target_names=['negative', 'positive']))

In [ ]:
!pip install transformers datasets torch scikit-learn -q
!pip uninstall torchvision -y -q

In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
print('kaz_clean' in dir())
print('pcm_clean' in dir())


True
True


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
label2id = {'negative': 0, 'positive': 1}
id2label = {0: 'negative', 1: 'positive'}

print("tokenizer, model_name, label2id, id2label ready")

tokenizer, model_name, label2id, id2label ready


In [ ]:
adaptation_data_hf = adaptation_data.copy()
adaptation_data_hf['label_id'] = adaptation_data_hf['label'].map(label2id)

adapt_ds = Dataset.from_pandas(adaptation_data_hf[['text', 'label_id']].reset_index(drop=True))

def tokenize_fn_adapt(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

adapt_ds = adapt_ds.map(tokenize_fn_adapt, batched=True)
adapt_ds = adapt_ds.rename_column('label_id', 'labels')
adapt_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

print(f"Adaptation dataset size: {len(adapt_ds)}")

adapt_class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=adaptation_data_hf['label_id'].values
)
adapt_class_weights = torch.tensor(adapt_class_weights, dtype=torch.float)
print(f"Adaptation class weights (negative, positive): {adapt_class_weights}")

adapt_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=2, id2label=id2label, label2id=label2id
)

class AdaptWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=adapt_class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

adapt_training_args = TrainingArguments(
    output_dir='./xlmr_adapted',
    num_train_epochs=1,
    per_device_train_batch_size=32,
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy='epoch',
    report_to='none',
)

adapt_trainer = AdaptWeightedTrainer(
    model=adapt_model,
    args=adapt_training_args,
    train_dataset=adapt_ds,
)

adapt_trainer.train()

adapt_trainer.save_model('./xlmr_adapted_final')
tokenizer.save_pretrained('./xlmr_adapted_final')

print("Stage 1 adaptation complete. Model saved to ./xlmr_adapted_final")

Map:   0%|          | 0/177993 [00:00<?, ? examples/s]

Adaptation dataset size: 177993
Adaptation class weights (negative, positive): tensor([2.4504, 0.6282])


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,0.614975
200,0.544019
300,0.504393
400,0.492323
500,0.575576
600,0.485420
700,0.446642
800,0.439824
900,0.431753
1000,0.448526


Step,Training Loss
100,0.614975
200,0.544019
300,0.504393
400,0.492323
500,0.575576
600,0.485420
700,0.446642
800,0.439824
900,0.431753
1000,0.448526


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Stage 1 adaptation complete. Model saved to ./xlmr_adapted_final


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copytree('./xlmr_adapted_final', '/content/drive/MyDrive/xlmr_adapted_final', dirs_exist_ok=True)
print("Backed up to Google Drive.")

Mounted at /content/drive
Backed up to Google Drive.


In [ ]:
# ==== STAGE 2: Fine-tune the Adapted Model on Nepal ====

# Reload Nepal HF datasets (reuse if still in memory, but rebuilding here for safety)
nepal_train_hf = nepal_train.copy()
nepal_valid_hf = nepal_valid.copy()
nepal_test_hf = nepal_test.copy()

for df in [nepal_train_hf, nepal_valid_hf, nepal_test_hf]:
    df['label_id'] = df['label'].map(label2id)

train_ds2 = Dataset.from_pandas(nepal_train_hf[['text', 'label_id']].reset_index(drop=True))
valid_ds2 = Dataset.from_pandas(nepal_valid_hf[['text', 'label_id']].reset_index(drop=True))
test_ds2 = Dataset.from_pandas(nepal_test_hf[['text', 'label_id']].reset_index(drop=True))

def tokenize_fn_nepal(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=256)

train_ds2 = train_ds2.map(tokenize_fn_nepal, batched=True)
valid_ds2 = valid_ds2.map(tokenize_fn_nepal, batched=True)
test_ds2 = test_ds2.map(tokenize_fn_nepal, batched=True)

for ds in [train_ds2, valid_ds2, test_ds2]:
    pass  # placeholder, rename happens next

train_ds2 = train_ds2.rename_column('label_id', 'labels')
valid_ds2 = valid_ds2.rename_column('label_id', 'labels')
test_ds2 = test_ds2.rename_column('label_id', 'labels')

train_ds2.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
valid_ds2.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
test_ds2.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

# Class weights — same as before, based on Nepal train
nepal_class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1]),
    y=nepal_train_hf['label_id'].values
)
nepal_class_weights = torch.tensor(nepal_class_weights, dtype=torch.float)
print(f"Nepal class weights (negative, positive): {nepal_class_weights}")

# ---- KEY DIFFERENCE: load the ADAPTED model, not fresh xlm-roberta-base ----
stage2_model = AutoModelForSequenceClassification.from_pretrained(
    './xlmr_adapted_final', num_labels=2, id2label=id2label, label2id=label2id
)

class Stage2WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=nepal_class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics_stage2(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

stage2_training_args = TrainingArguments(
    output_dir='./xlmr_stage2_nepal',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
)

stage2_trainer = Stage2WeightedTrainer(
    model=stage2_model,
    args=stage2_training_args,
    train_dataset=train_ds2,
    eval_dataset=valid_ds2,
    compute_metrics=compute_metrics_stage2,
)

stage2_trainer.train()

test_results2 = stage2_trainer.predict(test_ds2)
preds2 = np.argmax(test_results2.predictions, axis=-1)
labels2 = test_results2.label_ids

print("\n=== STAGE 2: ADAPTED XLM-R FINE-TUNED — Nepal test set ===")
print(f"Accuracy: {accuracy_score(labels2, preds2):.3f}")
print(f"F1 (weighted): {f1_score(labels2, preds2, average='weighted'):.3f}")
print(f"F1 (macro): {f1_score(labels2, preds2, average='macro'):.3f}")
print()
print(classification_report(labels2, preds2, target_names=['negative', 'positive']))

Map:   0%|          | 0/5345 [00:00<?, ? examples/s]

Map:   0%|          | 0/1146 [00:00<?, ? examples/s]

Map:   0%|          | 0/1146 [00:00<?, ? examples/s]

Nepal class weights (negative, positive): tensor([2.8014, 0.6086])


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== STAGE 2: ADAPTED XLM-R FINE-TUNED — Nepal test set ===
Accuracy: 0.969
F1 (weighted): 0.968
F1 (macro): 0.946

              precision    recall  f1-score   support

    negative       0.92      0.90      0.91       205
    positive       0.98      0.98      0.98       941

    accuracy                           0.97      1146
   macro avg       0.95      0.94      0.95      1146
weighted avg       0.97      0.97      0.97      1146



In [ ]:
import shutil

# Back up the Stage 2 model (adapted + fine-tuned on Nepal — your best result)
shutil.copytree('./xlmr_stage2_nepal', '/content/drive/MyDrive/xlmr_stage2_nepal', dirs_exist_ok=True)
print("Stage 2 model backed up to Google Drive.")


Stage 2 model backed up to Google Drive.
